
# Lantern Second Pass on Local ANTARES Data

Local version of `investigating_second_pass.ipynb`.

Workflow:

1. read cleaned loci and alerts;
2. keep alerts from **2026-05-27 onward**;
3. keep alerts containing `lsst_diaSource_band`;
4. select First-Cut loci with `max_score > 0.9691`;
5. flag and exclude inconsistent targets with `n_tagged > n_lsst_alerts`;
6. optionally sample the eligible loci (default: 10%);
7. build the target-level Second-Pass features;
8. apply `second_pass_first_test.pkl`;
9. select candidates with `second_pass_score > 0.8`;
10. save candidate loci and their alerts.

`n_detections` and `percent_tagged` follow the investigation notebook:
`n_detections = count(non-null x)` and
`percent_tagged = 100 * n_tagged / n_detections`.


In [ ]:

import os
import ast
import json
import pickle
from collections.abc import Mapping

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------- INPUTS --------------------

LOCI_FILE = "antares_data_clean_from_20260527/loci/loci_00000.parquet"
ALERTS_FILE = "antares_data_clean_from_20260527/alerts/alerts_00000.parquet"
MODEL_FILE = "second_pass_first_test.pkl"

# First-Cut consistency check used by investigating_second_pass.ipynb
FIRST_CUT_THRESHOLD = 0.9691

# Include alerts dated May 27, 2026 or later
START_MJD = 61187.0

# Run on 10% of eligible First-Cut loci.
# Set to 1.0 (or None) to run all eligible loci.
SAMPLE_FRAC = 0.10
RANDOM_STATE = 42

# Investigation notebook's working Second-Pass cut
SECOND_PASS_THRESHOLD = 0.80

OUTPUT_DIR = "second_pass_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:

# Load data and model

loci = pd.read_parquet(LOCI_FILE)
alerts = pd.read_parquet(ALERTS_FILE)

with open(MODEL_FILE, "rb") as f:
    model = pickle.load(f)

required_locus = {
    "locus_id", "ra", "dec",
    "max_score", "num_tagged_alerts",
}
required_alert = {
    "locus_id", "alert_id", "mjd", "alert_properties",
}

missing = required_locus - set(loci.columns)
if missing:
    raise KeyError(f"Missing locus columns: {sorted(missing)}")

missing = required_alert - set(alerts.columns)
if missing:
    raise KeyError(f"Missing alert columns: {sorted(missing)}")

if loci["locus_id"].duplicated().any():
    raise ValueError("LOCI_FILE contains duplicate locus_id rows.")

if alerts["alert_id"].duplicated().any():
    raise ValueError("ALERTS_FILE contains duplicate alert_id rows.")

print(f"Loci:   {len(loci):,}")
print(f"Alerts: {len(alerts):,}")
print(f"Model:  {MODEL_FILE}")


In [ ]:

# Helpers

def as_dict(value):
    if isinstance(value, Mapping):
        return value

    if isinstance(value, str):
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(value)
                if isinstance(parsed, Mapping):
                    return parsed
            except Exception:
                pass

    return {}


def get_prop(value, key, default=np.nan):
    value = as_dict(value)
    result = value.get(key, default)
    return default if result is None else result


def has_prop(value, key):
    return key in as_dict(value)


def normalize_band(series):
    # The investigation notebook defines this normalization but does not
    # explicitly call it. We apply it here so FilterLabel(...) and plain
    # "g"/"r"/... values are handled consistently.
    s = series.astype(str)

    extracted = s.str.extract(
        r"band=['\"]([ugrizy])['\"]",
        expand=False,
    )
    plain = s.str.extract(
        r"^([ugrizy])$",
        expand=False,
    )

    return extracted.fillna(plain)


## 1. Restrict the alert sample

In [ ]:

alerts = alerts.copy()
alerts["mjd"] = pd.to_numeric(alerts["mjd"], errors="coerce")

n_initial = len(alerts)

# Analysis is restricted to the confirmed Lantern operating period.
alerts = alerts[alerts["mjd"] >= START_MJD].copy()
n_after_date = len(alerts)

# Match investigating_second_pass.ipynb:
# only alerts carrying an LSST band enter the Second-Pass analysis.
has_band = alerts["alert_properties"].map(
    lambda p: has_prop(p, "lsst_diaSource_band")
)
alerts = alerts[has_band].copy()

print(f"Initial alerts:                    {n_initial:,}")
print(f"Alerts on/after MJD {START_MJD}:       {n_after_date:,}")
print(f"Alerts with lsst_diaSource_band:  {len(alerts):,}")

if len(alerts):
    print(f"MJD range used: {alerts['mjd'].min():.6f} - {alerts['mjd'].max():.6f}")


## 2. Select First-Cut loci and flag counter anomalies

In [ ]:

# The local 10k sample was already downloaded from the Lantern-tagged
# population, so no additional local tag check is needed.

first_cut = loci.copy()

first_cut["first_cut_max_score"] = pd.to_numeric(
    first_cut["max_score"],
    errors="coerce",
)
first_cut["n_tagged"] = pd.to_numeric(
    first_cut["num_tagged_alerts"],
    errors="coerce",
)

first_cut = first_cut[
    first_cut["first_cut_max_score"] > FIRST_CUT_THRESHOLD
].copy()

print(f"First-Cut loci: {len(first_cut):,}")


In [ ]:

# n_lsst_alerts = number of retained LSST alert rows for each locus.
# This is equivalent to the investigation notebook's local
# alert_id = 0,1,2,... followed by max(alert_id) + 1.

alert_counts = (
    alerts[alerts["locus_id"].isin(first_cut["locus_id"])]
    .groupby("locus_id")
    .size()
    .rename("n_lsst_alerts")
)

first_cut["n_lsst_alerts"] = (
    first_cut["locus_id"]
    .map(alert_counts)
    .fillna(0)
    .astype(int)
)

first_cut["missing_counter_flag"] = first_cut["n_tagged"].isna()
first_cut["bad_counter_flag"] = (
    first_cut["n_tagged"] > first_cut["n_lsst_alerts"]
)
first_cut["no_alert_flag"] = (
    first_cut["n_lsst_alerts"] == 0
)

flagged = first_cut[
    first_cut["missing_counter_flag"]
    | first_cut["bad_counter_flag"]
    | first_cut["no_alert_flag"]
].copy()

print(f"Missing n_tagged:        {first_cut['missing_counter_flag'].sum():,}")
print(f"n_tagged > n_lsst:       {first_cut['bad_counter_flag'].sum():,}")
print(f"No retained LSST alert:  {first_cut['no_alert_flag'].sum():,}")

if len(flagged):
    display(
        flagged[
            [
                "locus_id", "ra", "dec",
                "n_tagged", "n_lsst_alerts",
                "missing_counter_flag",
                "bad_counter_flag",
                "no_alert_flag",
            ]
        ]
    )

flagged.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "second_pass_flagged_targets.csv",
    ),
    index=False,
)

eligible = first_cut[
    ~first_cut["missing_counter_flag"]
    & ~first_cut["bad_counter_flag"]
    & ~first_cut["no_alert_flag"]
].copy()

print(f"Eligible First-Cut loci: {len(eligible):,}")


## 3. Sample eligible targets

In [ ]:

if SAMPLE_FRAC is None or SAMPLE_FRAC >= 1:
    targets = eligible.copy()
else:
    n_sample = max(
        1,
        int(round(len(eligible) * SAMPLE_FRAC)),
    )
    targets = eligible.sample(
        n=n_sample,
        random_state=RANDOM_STATE,
    ).copy()

target_ids = set(targets["locus_id"])

target_alerts = alerts[
    alerts["locus_id"].isin(target_ids)
].copy()

print(f"Targets used: {len(targets):,}")
print(f"Alerts used:  {len(target_alerts):,}")


## 4. Build alert-level quantities

In [ ]:

def build_alert_features(alert_subset):
    p = alert_subset["alert_properties"]

    keys = {
        "band": "lsst_diaSource_band",
        "x": "lsst_diaSource_x",
        "y": "lsst_diaSource_y",
        "xErr": "lsst_diaSource_xErr",
        "yErr": "lsst_diaSource_yErr",
        "ra": "lsst_diaSource_ra",
        "dec": "lsst_diaSource_dec",
        "scienceFlux": "lsst_diaSource_scienceFlux",
        "psfFlux": "lsst_diaSource_psfFlux",
        "apFlux": "lsst_diaSource_apFlux",
        "extendedness": "lsst_diaSource_extendedness",
        "dipoleLength": "lsst_diaSource_dipoleLength",
        "ixx": "lsst_diaSource_ixx",
        "iyy": "lsst_diaSource_iyy",
        "ixxPSF": "lsst_diaSource_ixxPSF",
        "iyyPSF": "lsst_diaSource_iyyPSF",
    }

    out = alert_subset[
        ["locus_id", "alert_id", "mjd"]
    ].copy()

    for name, key in keys.items():
        out[name] = p.map(
            lambda value, k=key: get_prop(value, k)
        )

    out["band"] = normalize_band(out["band"])

    numeric = [name for name in keys if name != "band"]
    out[numeric] = out[numeric].apply(
        pd.to_numeric,
        errors="coerce",
    )

    psf_trace = out["ixxPSF"] + out["iyyPSF"]
    src_trace = out["ixx"] + out["iyy"]

    out["moment_ext"] = np.where(
        psf_trace != 0,
        src_trace / psf_trace,
        np.nan,
    )

    out["template_flux"] = (
        out["scienceFlux"] - out["psfFlux"]
    )

    out["flux_ext"] = np.where(
        out["psfFlux"] != 0,
        out["apFlux"] / out["psfFlux"],
        np.nan,
    )

    out["x_y_err"] = np.sqrt(
        out["xErr"]**2 + out["yErr"]**2
    )

    return out


df = build_alert_features(target_alerts)

print(f"Alert-level rows: {len(df):,}")
df.head()


## 5. Build one Second-Pass row per locus

In [ ]:

def calculate_stats(df, target_meta):
    grouped = df.groupby("locus_id")

    summary = grouped.agg(
        # Investigation notebook definitions
        n_detections=("x", "count"),
        n_lsst_alerts=("alert_id", "size"),

        x_std=("x", "std"),
        y_std=("y", "std"),
        mean_x_y_err=("x_y_err", "mean"),
        median_x_y_err=("x_y_err", "median"),

        moment_ext_mean=("moment_ext", "mean"),
        moment_ext_std=("moment_ext", "std"),
        extendedness_mean=("extendedness", "mean"),
        extendedness_std=("extendedness", "std"),
        dipoleLength_mean=("dipoleLength", "mean"),
        dipoleLength_std=("dipoleLength", "std"),

        ra=("ra", "first"),
        dec=("dec", "first"),

        # Useful output metadata
        mjd_min=("mjd", "min"),
        mjd_max=("mjd", "max"),
    ).reset_index()

    summary["centroid_std"] = np.sqrt(
        summary["x_std"]**2
        + summary["y_std"]**2
    )

    summary["centroid_instability"] = np.where(
        summary["median_x_y_err"] > 0,
        summary["centroid_std"]
        / summary["median_x_y_err"],
        0.0,
    )

    # Same single-detection handling as investigation notebook
    summary["centroid_std"] = (
        summary["centroid_std"].fillna(0.0)
    )
    summary["centroid_instability"] = (
        summary["centroid_instability"].fillna(0.0)
    )

    for band in "ugrizy":
        band_data = (
            df[df["band"] == band]
            .groupby("locus_id")
            .agg(
                **{
                    f"apFlux_mean_{band}": (
                        "apFlux", "mean"
                    ),
                    f"apFlux_std_{band}": (
                        "apFlux", "std"
                    ),
                    f"template_flux_mean_{band}": (
                        "template_flux", "mean"
                    ),
                    f"flux_ext_mean_{band}": (
                        "flux_ext", "mean"
                    ),
                    f"flux_ext_std_{band}": (
                        "flux_ext", "std"
                    ),
                }
            )
        )

        summary = summary.merge(
            band_data,
            left_on="locus_id",
            right_index=True,
            how="left",
        )

    meta = target_meta[
        [
            "locus_id", "ra", "dec",
            "first_cut_max_score",
            "n_tagged",
        ]
    ].rename(
        columns={
            "ra": "locus_ra",
            "dec": "locus_dec",
        }
    )

    summary = summary.merge(
        meta,
        on="locus_id",
        how="left",
    )

    # Use alert coordinates as in the investigation notebook;
    # fall back to the locus coordinates if needed.
    summary["ra"] = (
        summary["ra"].fillna(summary["locus_ra"])
    )
    summary["dec"] = (
        summary["dec"].fillna(summary["locus_dec"])
    )

    # Investigation notebook definition
    summary["percent_tagged"] = np.where(
        summary["n_detections"] > 0,
        np.round(
            100.0
            * summary["n_tagged"]
            / summary["n_detections"],
            5,
        ),
        np.nan,
    )

    return summary


summary = calculate_stats(df, targets)

# Diagnostic only: n_detections and n_lsst_alerts are expected to
# be the same when x is present for every retained LSST alert.
count_mismatch = (
    summary["n_detections"]
    != summary["n_lsst_alerts"]
)

print(f"Second-Pass rows: {len(summary):,}")
print(
    "n_detections != n_lsst_alerts: "
    f"{count_mismatch.sum():,}"
)

if count_mismatch.any():
    display(
        summary.loc[
            count_mismatch,
            [
                "locus_id",
                "n_tagged",
                "n_detections",
                "n_lsst_alerts",
                "percent_tagged",
            ],
        ]
    )

# A target with zero valid x values cannot reproduce the
# investigation notebook's percent_tagged definition.
zero_detection = summary["n_detections"] == 0

if zero_detection.any():
    display(
        summary.loc[
            zero_detection,
            [
                "locus_id",
                "n_tagged",
                "n_detections",
                "n_lsst_alerts",
            ],
        ]
    )

summary = summary[~zero_detection].copy()

print(f"Targets retained for scoring: {len(summary):,}")
summary.head()


## 6. Apply the Second-Pass model

In [ ]:

# Feature order produced by investigating_second_pass.ipynb
DEFAULT_MODEL_FEATURES = [
    "x_std",
    "y_std",
    "mean_x_y_err",
    "median_x_y_err",
    "moment_ext_mean",
    "moment_ext_std",
    "extendedness_mean",
    "extendedness_std",
    "dipoleLength_mean",
    "dipoleLength_std",
    "centroid_std",
    "centroid_instability",

    "apFlux_mean_u",
    "apFlux_std_u",
    "template_flux_mean_u",
    "flux_ext_mean_u",
    "flux_ext_std_u",

    "apFlux_mean_g",
    "apFlux_std_g",
    "template_flux_mean_g",
    "flux_ext_mean_g",
    "flux_ext_std_g",

    "apFlux_mean_r",
    "apFlux_std_r",
    "template_flux_mean_r",
    "flux_ext_mean_r",
    "flux_ext_std_r",

    "apFlux_mean_i",
    "apFlux_std_i",
    "template_flux_mean_i",
    "flux_ext_mean_i",
    "flux_ext_std_i",

    "apFlux_mean_z",
    "apFlux_std_z",
    "template_flux_mean_z",
    "flux_ext_mean_z",
    "flux_ext_std_z",

    "apFlux_mean_y",
    "apFlux_std_y",
    "template_flux_mean_y",
    "flux_ext_mean_y",
    "flux_ext_std_y",

    "n_tagged",
    "percent_tagged",
]


def get_model_features(model):
    if hasattr(model, "feature_names_in_"):
        return list(model.feature_names_in_)

    try:
        names = model.get_booster().feature_names
        if names:
            return list(names)
    except Exception:
        pass

    return DEFAULT_MODEL_FEATURES


model_features = get_model_features(model)

missing = [
    name
    for name in model_features
    if name not in summary.columns
]
if missing:
    raise KeyError(
        "Model requires features not constructed here:\n"
        + "\n".join(missing)
    )

X = summary[
    model_features
].replace(
    [np.inf, -np.inf],
    np.nan,
)

print(f"Model input features: {len(model_features)}")

summary["second_pass_score"] = (
    model.predict_proba(X)[:, 1]
)

summary["candidate_flag"] = (
    summary["second_pass_score"]
    > SECOND_PASS_THRESHOLD
)

candidates = (
    summary[summary["candidate_flag"]]
    .sort_values(
        "second_pass_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    f"Candidates: {len(candidates):,} / {len(summary):,} "
    f"(score > {SECOND_PASS_THRESHOLD})"
)


In [ ]:

candidate_columns = [
    "locus_id",
    "ra",
    "dec",
    "first_cut_max_score",
    "n_tagged",
    "n_detections",
    "n_lsst_alerts",
    "percent_tagged",
    "mjd_min",
    "mjd_max",
    "second_pass_score",
]

candidates[candidate_columns]


## 7. Save outputs

In [ ]:

candidate_ids = set(
    candidates["locus_id"]
)

candidate_alerts = target_alerts[
    target_alerts["locus_id"].isin(candidate_ids)
].copy()

summary.to_parquet(
    os.path.join(
        OUTPUT_DIR,
        "second_pass_scored_targets.parquet",
    ),
    index=False,
)

candidates[candidate_columns].to_csv(
    os.path.join(
        OUTPUT_DIR,
        "second_pass_candidates.csv",
    ),
    index=False,
)

candidates[candidate_columns].to_parquet(
    os.path.join(
        OUTPUT_DIR,
        "second_pass_candidates.parquet",
    ),
    index=False,
)

candidate_alerts.to_parquet(
    os.path.join(
        OUTPUT_DIR,
        "second_pass_candidate_alerts.parquet",
    ),
    index=False,
)

print("Saved:")
print("  second_pass_flagged_targets.csv")
print("  second_pass_scored_targets.parquet")
print("  second_pass_candidates.csv")
print("  second_pass_candidates.parquet")
print("  second_pass_candidate_alerts.parquet")


## Optional: score distribution

In [ ]:

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

plt.figure(figsize=(4, 3))

plt.hist(
    summary["second_pass_score"],
    bins=30,
    histtype="step",
)

plt.axvline(
    SECOND_PASS_THRESHOLD,
    ls="--",
)

plt.xlabel("Second-Pass score")
plt.ylabel("Targets")
plt.tight_layout()
plt.show()


In [ ]:
p = summary["second_pass_score"]

print(p.describe())

for x in [0.001, 0.01, 0.05, 0.1, 0.9, 0.95, 0.99, 0.999]:
    if x < 0.5:
        print(f"score < {x:5.3f}: {(p < x).sum():4d}  ({(p < x).mean():.1%})")
    else:
        print(f"score > {x:5.3f}: {(p > x).sum():4d}  ({(p > x).mean():.1%})")

In [ ]:
print("Exactly 0:", (p == 0).sum())
print("Exactly 1:", (p == 1).sum())

print("Minimum:", p.min())
print("Maximum:", p.max())

In [ ]:
eps = 1e-6
p_clip = np.clip(p, eps, 1 - eps)

logit_p = np.log(
    p_clip / (1 - p_clip)
)

plt.figure(figsize=(4, 3))
plt.hist(logit_p, bins=30, histtype="step")
plt.xlabel("logit(Second-Pass score)")
plt.ylabel("Targets")
plt.tight_layout()
plt.show()


### Notes

- The default model is exactly the one loaded by `investigating_second_pass.ipynb`: `second_pass_first_test.pkl`.
- The local 10k loci were already selected from the Lantern-tagged population; no redundant local tag check is repeated.
- The original `max_score > 0.9691` consistency check is retained.
- The May 27 restriction is an analysis requirement for this local dataset.
- Alerts must contain `lsst_diaSource_band`, matching the investigation notebook.
- Band values are explicitly normalized before per-band aggregation.
- `n_detections = count(non-null x)` and `percent_tagged = 100*n_tagged/n_detections`, matching the investigation notebook.
- `n_detections != n_lsst_alerts` is reported as a diagnostic only.
- Targets with `n_tagged > n_lsst_alerts` are flagged and excluded from follow-up tests.
- Candidate selection is `second_pass_score > 0.8`, matching the investigation notebook.
